#Inference Notebook
**Attention**: This notebook uses shell commands for MIDI to CP representation. Please specify the right paths for MidiBERT's data conversion pipeline meaning the ".../data_creation/prepare_data/main.py" and ".../data_creation/prepare_data/dict/CP.pkl". You may do this via searching "!python3" keyword to find where we've used the shell commands.  

In [ ]:
#@title Downloading Required Packages
from IPython.display import clear_output

!pip install transformers==4.29.2
!pip install optimum
!pip install miditok==2.1.2
!pip install loguru
!pip install pretty_midi
!sudo apt install -y fluidsynth
!pip install --upgrade pyfluidsynth
!pip install faiss-cpu
clear_output()
print("Downloads done succesfully!")

Downloads done succesfully!


In [ ]:
#@title Importing Required Libraries

from transformers import GenerationConfig, GPT2LMHeadModel
import torch
from torch.utils.data import DataLoader, Dataset
from typing import List, Dict, Optional, Union, Tuple, Any
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
from miditok import REMI
from loguru import logger as lg
import json
import pickle
import os
import random
from miditok import MIDITokenizer
from miditoolkit import MidiFile
import miditoolkit
import tempfile
import numpy as np
import faiss
import sys
sys.path.append("/MIDI_BERT_CP/")

from process_search import ProcessDataset, SimilaritySearch

import pretty_midi as pm
import collections
import pandas as pd
import fluidsynth
import math
import matplotlib.pyplot as plt
import numpy as np
import time
import seaborn as sns
from IPython import display

lg.info("Imports done succesfully!")

2023-08-19 16:06:59.994 | INFO     | __main__:<cell line: 39>:39 - Imports done succesfully!


##Utility Functions
These are the utility functions for displaying midi data and inference.

In [ ]:
#@title For Displaying Statistics
class DisplayStats:
    """This class is for the displaying statistics of the generated MIDI.
    It's possible to,
     - Listen the audio on Colab,
     - See the pianaroll,
     - See the pitch, duration, step.

     @Params
     @path_to_midi (Optional): path to the generated midi file.
     @all: show all possible statistics.
    """

    def __init__(self, path_to_midi: Optional[Union[str, Path]], all_data:bool=True) -> None:
        if path_to_midi != None:
            if isinstance(path_to_midi, Path):
                path_to_midi = str(path_to_midi)
            self.pm = pm.PrettyMIDI(path_to_midi)
        else:
            pass

        self.raw_notes = self.midi_to_notes(self.pm)
        self.raw_notes.head()

        if all_data:
            self.display_audio(self.pm)
            self.plot_piano_roll(self.raw_notes, count=10000)
            self.plot_distributions(self.raw_notes)

    def display_audio(
        self,
        pm: Optional[pm.PrettyMIDI]=None,
        _SAMPLING_RATE: int = 44100,
    ):
        """Displaying audio in Colab Environment"""
        if pm == None:
            pm = self.pm
        seconds = pm.get_end_time()
        waveform = pm.fluidsynth(fs=_SAMPLING_RATE)
        # Take a sample of the generated waveform to mitigate kernel resets
        waveform_short = waveform[: math.floor(seconds * _SAMPLING_RATE)]
        return display.Audio(waveform_short, rate=_SAMPLING_RATE)

    def midi_to_notes(self, pm: Optional[pm.PrettyMIDI]) -> pd.DataFrame:
        if pm == None:
            pm = self.pm

        instrument = pm.instruments[0]
        notes = collections.defaultdict(list)

        # Sort the notes by start time
        sorted_notes = sorted(instrument.notes, key=lambda note: note.start)
        prev_start = sorted_notes[0].start

        for note in sorted_notes:
            start = note.start
            end = note.end
            notes["pitch"].append(note.pitch)
            notes["start"].append(start)
            notes["end"].append(end)
            notes["step"].append(start - prev_start)
            notes["duration"].append(end - start)
            prev_start = start

        return pd.DataFrame({name: np.array(value) for name, value in notes.items()})

    def plot_piano_roll(self, notes: pd.DataFrame, count: int = 100):
        if count:
            title = f"First {count} notes"
        else:
            title = f"Whole track"
            count = len(notes["pitch"])
        plt.figure(figsize=(20, 4))
        plot_pitch = np.stack([notes["pitch"], notes["pitch"]], axis=0)
        plot_start_stop = np.stack([notes["start"], notes["end"]], axis=0)
        plt.plot(
            plot_start_stop[:, :count], plot_pitch[:, :count], color="b", marker="."
        )
        plt.xlabel("Time [s]")
        plt.ylabel("Pitch")
        _ = plt.title(title)

    def plot_distributions(self, notes: pd.DataFrame, drop_percentile=2.5):
        plt.figure(figsize=[15, 5])
        plt.subplot(1, 3, 1)
        sns.histplot(notes, x="pitch", bins=20)

        plt.subplot(1, 3, 2)
        max_step = np.percentile(notes["step"], 100 - drop_percentile)
        sns.histplot(notes, x="step", bins=np.linspace(0, max_step, 21))

        plt.subplot(1, 3, 3)
        max_duration = np.percentile(notes["duration"], 100 - drop_percentile)
        sns.histplot(notes, x="duration", bins=np.linspace(0, max_duration, 21))


In [ ]:
#@title For Inference
def collate_gen_left(batch: List[Dict[str, torch.LongTensor]]) -> torch.LongTensor:
    """Here the sequences are padded to the left, so that the last token along the time dimension
    is always the last token of each seq, allowing to efficiently generate by batch."""
    bos_shape = (1,)
    #lg.debug(f"THE BATCH: {batch}")
    batch = [
        torch.flip(
            torch.cat([torch.full(bos_shape, BOS_None), seq["input_ids"]], dim=0),
            dims=(0,),
        )
        for seq in batch
    ]
    batch = torch.nn.utils.rnn.pad_sequence(
        batch, batch_first=True, padding_value=PAD_None
    )  # (N,T) or (N,T,Z)
    batch = torch.flip(batch, dims=(1,)).long()
    return batch  # (N,T)


class InferenceInput(Dataset):
    """Preparing inputs for the inference stage. This class is used with a DataLoader
    in order to batching. But I might need to change that in the future.
    @Params:
    @path: path to a .json which consists of dumped tokens. Expected to be List[List[int]]
    @tokens: it's possible to give tokens directly as well. Processing the suitable from will
    later be done in the following cells. {"input_ids":[torch.Tensor(...)], "labels":[torch.Tensor(...)]}
    """

    def __init__(
        self,
        path: Optional[Union[str, Path]] = None,
        tokens: Optional[List[List[int]]] = None,
    ) -> None:
        super().__init__()
        if path == None and tokens == None:
            lg.critical("You should at least spesicify one of the input parameters.")
            raise Exception
        elif path != None and tokens == None:
            with open(path) as json_file:
                self.samples = json.load(json_file)
        elif path == None and tokens != None:
            self.samples = tokens
        else:  # both spesified.
            with open(path) as json_file:
                self.samples = json.load(json_file) + tokens
        # self.samples = []

    def __getitem__(self, idx) -> Dict[str, torch.LongTensor]:
        if isinstance(self.samples[idx], list):
          tensor_sample = torch.tensor(self.samples[idx])
        return {"input_ids": tensor_sample, "labels": tensor_sample}

    def __len__(self) -> int:
        return len(self.samples)

    def __repr__(self):
        return self.__str__()

    def __str__(self) -> str:
        return "No data loaded" if len(self) == 0 else f"{len(self.samples)} samples"


##Inference Phase

In [ ]:
#@title Configuration Parameters
PAD_None = 0
BOS_None = 1
EOS_None = 2
TOKENIZER_PATH = ".../samples/config_REMI.txt"  # this should be the params.json file
MAIN_PATH = Path(".../structure/") # will create new directories in it. Please refer to GenerateWithRetrieval class. 
GPT_CKPT_PATH = ".../GPT_checkpoints/Aug15_1/checkpoint-11000/"
DATALOADER_BATCH_SIZE=1

# Each of these parameters can be set to None to disable.
INFERENCE_DATA_PATH = None
DATA_SPLIT = None
INFERENCE_TOKENS = [1]

# These parameters will directly be fed to GenerateWithRetrieval class.
# Please specify the ...data_creation/prepare_data/main.py path.
CP_VOCAB_PATH = ".../MIDI_BERT_CP/data_creation/prepare_data/dict/CP.pkl"
MIDIBERT_CKPT_PATH = ".../MIDI_BERT_CP/CHECKPOINT/pretrain_model.ckpt" # checkpoint can be downloaded from MidiBERT's github page.
EMBEDDING_DATALOADER_BATCH_SIZE = 1

generation_config = GenerationConfig(
    max_new_tokens=256,  # extends samples by n tokens
    do_sample=True, 
    temperature=1,
    top_k=15,
    top_p=0.9,
    epsilon_cutoff=3e-4,
    eta_cutoff=1e-3,
    pad_token_id=PAD_None,
    bos_token_id=BOS_None,
    eos_token_id=EOS_None
)


In [ ]:
#@title Initializing the pretrained model and the dataset
model = GPT2LMHeadModel.from_pretrained(GPT_CKPT_PATH)
tokenizer = REMI(params=TOKENIZER_PATH)

inference_data = InferenceInput(path=INFERENCE_DATA_PATH, tokens=[INFERENCE_TOKENS])
inference_data = (
    torch.utils.data.Subset(
        inference_data, list(range(0, len(inference_data) * DATA_SPLIT))
    )
    if DATA_SPLIT is not None
    else inference_data
)

dataloader_test = DataLoader(inference_data, batch_size=DATALOADER_BATCH_SIZE, collate_fn=collate_gen_left)


In [ ]:
#@title Class for retrieval
class GenerateWithRetrieval:
    """Process the user-defined-midi-dataset and creates directories and files accordingly to use retrieval approaches while generating.

    @Params:
    ---
    main_output_path
inference_data
generation_config
tokenizer
prepare_retrieval
dir_dict
user_defined_dataset_path
    
    * main_output_path: will save everything to corresponding folders. So it's advised to specify an empty folder to use.
    * inference_data: InferenceInput object to use.
    * generation_config: generation config to use. 
    * tokenizer: one of Miditok Tokenizer classes. Possibly REMI with config.txt already loaded.
    * prepare_retrieval: Creates all the necessary files into the already constructed folders.
    * user_defined_dataset_path: path to the midi files of user defined dataset. glob will be made afterwards.
    * dir_dict: directory dictionary. Should include whatever in the non-initialized version of this dictionary plus user_dataset_CP_npy_path.
    user_dataset_CP_npy_path is the .npy file that has the tokenized with CP version of the dataset. Not to be confused with embeds_CPtokens.
    embeds_CPtokens is a dictionary with shape: {0: (torch.Tensor(EMBEDDINGS_0), RELATIVE_TOKENS_CP_0), 1:(torch.Tensor(EMBEDDINGS_1), RELATIVE_TOKENS_CP_1), ...}
    * user_defined_dataset_path: This is the path of the midi files you have. Will be used when creating the necessary files for the first time. Then it's advised
    to specify this path in the dir_dict.

        Here is a dir_dict template to use:

        ```
        dir_dict = {
                "user_defined_dataset": Path("..."), # folder of user-defined MIDI files
                "samples": Path("..."), # folder of the generated midi files.
                "index": Path("..."), # NOT a folder, files path of the index.
                "user_dataset_CP": Path("..."), # folder to put CP tokenized data
                "user_dataset_CP_npy_path": Path("..."), # NOT a folder, it's the files' path.
                "embeddings": Path("..."), # folder of the embeddings
                "user_dataset_MidiBERTembeds_CPtokens_ALL" : Path("..."), # NOT a folder, it's the files' path.
            }

        ```

    Guideline on how to use this class:
    ---
    1) If it's the first time working with the user selected songs that you have, then run the prepare_retrieval() function or just set
    prepare_retrieval=True.
    This function will create every file which you need to generate, into it's own folder using not-initialized-by-you self.dir_dict.

    2) If you've already created the necessary files and you wish to generate with retrieval with an input, you
    should pass each of the files path to dir_dict param. This way you won't need to process the user selected songs again.

    Notes
    ---
    * Please look at the CONSTANTS which is defined previously. I've used them directly here, so this class may not
    work properly without them. I tried to give place the constants that has been used in a function in the function descriptions.
    """

    def __init__(
        self,
        main_output_path: Union[str, Path],
        inference_data: InferenceInput,
        generation_config: GenerationConfig,
        tokenizer: Optional[MIDITokenizer] = None,
        prepare_retrieval: bool = False,
        dir_dict: Optional[Dict[str, Path]] = None,
        user_defined_dataset_path: Optional[Path] = None,
    ) -> None:
        self.generation_config = generation_config
        self.main_output_path = main_output_path
        self.inference_data = inference_data
        self.tokenizer = tokenizer

        self.user_data_REMI: List[List[int]] = []

        # Instantiating the processor, more likely the model. This processor will only be used with preprare_processor()
        # to create an instance of it and fill it with data.
        self.processor = ProcessDataset(
            vocabPath=CP_VOCAB_PATH,
            dataPath=None,
            dataItself=None,
            checkpointPath=MIDIBERT_CKPT_PATH,
            verbose=False,
            div=1,
            pool_type="avg",
        )

        self.similarity_searcher: Optional[SimilaritySearch] = None
        self.index: Optional[faiss.IndexFlatIP] = None

        with open(CP_VOCAB_PATH, "rb") as CP_dict_file:
            self.CP_dict_e2w_w2e = pickle.load(CP_dict_file)

        if dir_dict == None:
            self.dir_dict = {
                "user_defined_dataset": user_defined_dataset_path,
                "samples": None,
                "user_dataset_CP": None,
                "embeddings": None,
                "index": None,
            }
            self.dir_dict = self._create_directories(
                self.main_output_path, self.dir_dict
            )
            if prepare_retrieval:
                # self.user_dataset_MidiBERTembeds_CPtokens_ALL is created at this function.
                self.prepare_retrieval()
        else:
            self.dir_dict = dir_dict
            # A big file to keep in RAM but I guess we'll manage. It's a must.
            # Also loading this comes with a form (array({int: (tensor,tensor)}))
            # So I'm removing the array stuff at the beginning and turning them to dict.
            self.user_dataset_MidiBERTembeds_CPtokens_ALL: Dict[
                int, Tuple[torch.Tensor, torch.Tensor]
            ] = np.load(
                # will throw keyerror if user didn't specify the relative key.
                self.dir_dict["user_dataset_MidiBERTembeds_CPtokens_ALL"],
                allow_pickle=True,
            ).tolist()

            lg.trace(self.user_dataset_MidiBERTembeds_CPtokens_ALL)
            lg.trace(type(self.user_dataset_MidiBERTembeds_CPtokens_ALL))

            self.index = faiss.read_index(str(self.dir_dict["index"]))
            if prepare_retrieval:
                lg.critical(
                    "You have already specified the required elements, why prepare retrieval again?"
                )
                raise Exception
        
        if prepare_retrieval:
            self.prepare_retrieval()

    def prepare_processor(
        self,
        dataPath: Optional[str] = None,
        dataItself: Optional[List[List[List[int]]]] = None,
    ):
        """Processor is already instantiated in the __init__() but it doesn't have any data in it.
        This function aims to fill it with data and reshapes it to be ready for __getitem__ in other
        words get and pool embeddings."""

        if dataPath != None:
            if Path(dataPath).exists() == False:
                lg.critical("Data doesn't exists.")
                raise Exception

        processor = self.processor
        processor.data = processor.instantiate_data(
            dataPath=dataPath, dataItself=dataItself
        )
        processor.div_reshape()

        return processor

    def prepare_retrieval(self):
        """Tokenizes given midi pieces with CP and saves them, creates MidiBERT embeddings (both in batches and merged) and saves them,
        creates an index to use in similarity search with the created embeddings and saves it. This created index will be used later by
        generation functions.

        No parameters here, just using: self.dir_dict["user_dataset_CP"], CP_VOCAB_PATH, MIDIBERT_CKPT_PATH, self.dir_dict["embeddings"],
        EMBEDDING_DATALOADER_BATCH_SIZE, self.dir_dict["index"].

        """
        user_dataset_CP_npy_path = self.dir_dict["user_dataset_CP"] / Path(
            "CP_tokenized_user_dataset.npy"
        )
        lg.info("Now tokenizing the user defined dataset.")
        self.tokenize_multiple_CP_save(
            input_path=str(self.dir_dict["user_defined_dataset"]),
            output_path=str(self.dir_dict["user_dataset_CP"]),
            name="CP_tokenized_user_dataset.npy",
        )

        lg.info("Preparing the processor")
        processor = self.prepare_processor(str(user_dataset_CP_npy_path), None)

        my_dataloader = DataLoader(
            processor,
            batch_size=EMBEDDING_DATALOADER_BATCH_SIZE,
        )
        # Calculating all the embeddings at once is a heavy task.
        # So we are saving them in batches. But we'll merge them later on.

        lg.info("Now saving the embeddings piece by piece.")
        for i_batch, item in enumerate(tqdm(my_dataloader)):
            # np.save(EMBEDDINGS_PATH + f"embedding{i_batch}.npy", item.detach().numpy())
            np.save(
                self.dir_dict["embeddings"] / Path(f"embedding_CPtoken{i_batch}.npy"),
                item,
                allow_pickle=True,
            )

        embeddings_path = self.dir_dict["embeddings"] / Path(
            "user_dataset_MidiBERTembeds_CPtokens_ALL.npy"
        )

        lg.info("Now merging the embeddings pieces.")
        # Here we are merging the embeddings, the whole process is for this variable.
        self.user_dataset_MidiBERTembeds_CPtokens_ALL = processor.merge_embeddings_v2(
            pathToEmbeddings=self.dir_dict["embeddings"],
            save=True,
            outputDir=embeddings_path,
        )
        self.dir_dict["user_dataset_MidiBERTembeds_CPtokens_ALL"] = embeddings_path

        self.similarity_searcher = SimilaritySearch(str(embeddings_path))

        lg.info("Now creating the cos index.")
        Cosindex = self.similarity_searcher.create_cos_index(
            self.similarity_searcher.embds, save=True, outputPath=self.dir_dict["index"]
        )

        self.index = Cosindex

    def REMI2MIDI2CP(self, tokens: List[int]) -> List[List[int]]:
        """REMI to CP representation using /temp folders.
        We are using SHELL COMMANDS here.
        """
        lg.debug("Now converting REMI2MIDI2CP. ")
        temp_path = tempfile.gettempdir()
        temp_path_REMImidi = temp_path + "/REMI2MIDI2CP_temp_REMImidi"

        self.tokenizer(tokens).dump(temp_path_REMImidi + ".mid")

        if Path(temp_path_REMImidi+ ".mid").exists() == False:
            lg.critical(
                f"REMI2MIDI2CP_temp_REMImidi.mid couldn't saved to {temp_path_REMImidi}."
            )
            raise Exception("Temporary midi couldn't saved.")

        lg.trace("Now walking into the temp folder...")

        for elem in os.walk(temp_path):
            lg.trace(elem)

        name = "REMI2MIDI2CP_temp_CP"
        temp_path_npy = temp_path
        !python3 /content/drive/MyDrive/tubitak2204A_2023_v2/MIDI_BERT_CP/data_creation/prepare_data/main.py --input_dir=$temp_path --output_dir=$temp_path_npy --name=$name --dict=$CP_VOCAB_PATH # type:ignore

        if Path(temp_path_npy + "/REMI2MIDI2CP_temp_CP.npy").exists() == True:
            lg.debug(
                f"Temp CP data is saved to {temp_path_npy + '/REMI2MIDI2CP_temp_CP.npy'}"
            )
        else:
            lg.critical(
                "Couldn't convert REMI2MIDI2CP because REMI2MIDI2CP_temp_CP.npy couldn't created or couln't find. This might be because of an error that CP tokenization throws."
            )
            raise Exception

        return np.load(temp_path_npy + "/REMI2MIDI2CP_temp_CP.npy", allow_pickle=True)

    def CP2MIDI2REMI(
        self,
        tokens: List[List[int]],
        CP_word2event: Optional[Dict[str, Dict[int, str]]] = None,
        tokenizer: Optional[MIDITokenizer] = None,
        _tempo: Optional[int] = 120,
    ):
        """This functions core is taken from https://github.com/wazenmai/MIDI-BERT/issues/10
        @Params:
        @tokens: tokens from CP representatation. E. g. Pitch 5, Duration 7, Bar New, ...
        @CP_word2event: Dictionary of CP. Original is saved as tuple(e2w, w2e)
        @tokenizer: REMI tokenizer

        """
        # Default resol parameters can be found in the link below.
        # https://github.com/wazenmai/MIDI-BERT/blob/CP/melody_extraction/midibert/utils.py

        tokenizer = self.tokenizer if tokenizer == None else tokenizer
        CP_word2event = (
            self.CP_dict_e2w_w2e[1] if CP_word2event == None else CP_word2event
        )

        TICK_RESOL = 480
        BAR_RESOL = 4 * TICK_RESOL

        class_keys = CP_word2event.keys()
        CP2MIDI_obj = miditoolkit.midi.parser.MidiFile()
        CP2MIDI_obj.tempo_changes = [
            miditoolkit.midi.containers.TempoChange(_tempo, 0.0)
        ]
        bar_cnt = -1

        all_notes = []

        for i in range(len(tokens)):
            vals = []
            for kidx, key in enumerate(class_keys):
                vals.append(CP_word2event[key][tokens[i][kidx]])
            # print(vals)

            if vals[0].split(" ")[-1] == "New":
                bar_cnt += 1
            if (vals[0].split(" ")[-1] == "New") or (
                vals[0].split(" ")[-1] == "Continue"
            ):
                position_ = vals[1].split(" ")[-1]
                if position_ != "<PAD>" and position_ != "<MASK>":
                    position = int(position_.split("/")[0])  # - 1
                else:
                    continue

                duration_ = vals[3].split(" ")[-1]
                if duration_ != "<PAD>" and duration_ != "<MASK>":
                    duration = (int(duration_) + 1) * 60
                    # Durations are Duration 0, Duration 1, ... , Duration 63. => 60 * 64 = 3840 ticks.
                    # Duration 64 is PAD, Duration 65 is MASK.
                    # Because 0*60 = 0, we are adding 1.
                else:
                    continue
                pitch_ = vals[2].split(" ")[-1]
                if pitch_ != "<PAD>" and pitch_ != "<MASK>":
                    pitch = int(pitch_)
                else:
                    continue
                st = bar_cnt * BAR_RESOL + position * TICK_RESOL
                et = st + duration
                all_notes.append(
                    miditoolkit.Note(velocity=80, pitch=pitch, start=st, end=et)
                )

        # Create the CP2MIDI
        guitar_track = miditoolkit.Instrument(24, is_drum=False)
        guitar_track.notes = all_notes
        CP2MIDI_obj.instruments = [guitar_track]

        # Tokenize REMI and return.
        return tokenizer(CP2MIDI_obj)[0].ids

    def tokenize_single_CP(self, single_midi_path: Union[str, Path, List[int]]):
        """Using a /temp folder, we save and load single CP representation.
        We are using SHELL COMMANDS here.
        @Params
        @single_midi_path: this could be a path to a MIDI file or it could be REMI tokens
        """

        temp_path_npy = tempfile.gettempdir() + "single_midi_CP.npy"
        !python3 .../MIDI_BERT_CP/data_creation/prepare_data/main.py --input_dir=$single_midi_path --output_dir=$temp_path_npy --dict=$CP_VOCAB_PATH  # type:ignore
        return np.load(temp_path_npy, allow_pickle=True)

    def tokenize_multiple_CP_save(self, input_path: str, output_path: str, name=str):
        """Tokenizes mulitple midi files in the *input_paths* using CP command line usage.

        We are using SHELL COMMANDS here.
        """

        !python3 /content/drive/MyDrive/tubitak2204A_2023_v2/MIDI_BERT_CP/data_creation/prepare_data/main.py --input_dir=$input_path --output_dir=$output_path --dict=$CP_VOCAB_PATH --name=$name # type:ignore

    def tokenize_multiple_REMI(self):
        """Tokenize the data using REMI tokenization and the constants specified.
        tokenizer.tokenize_midi_dataset() can not be used here because it only saves the tokens,
        while we will be taking the all datas into RAM. Yeah, it doesn't sound so nice but we are not
        expecting many user_defined_songs.

        No parameters, using self.tokenizer, self.dir_dict["user_defined_dataset"], self.user_data_REMI
        """

        if self.tokenizer == None:
            lg.critical("You should specify a tokenizer to tokenize. Makes sense?")
            raise Exception

        for path in self.dir_dict["user_defined_dataset"].glob("*.mid"):
            self.user_data_REMI.append(
                self.tokenizer(path)[0].ids
            )  # might throw error.

    def generate_with_retrieval_probability(
        self,
        single_input: torch.Tensor,
        probability: float,
        model: GPT2LMHeadModel,
        repetition_search_div: int = 6,
    ):
        """While generating, retrieve similar datas with a probability. This function doens't
        have a continuosity, for continuous generation with retrieval please refer to cont_gen_retrieval()
        After retrieving the most similar piece, we take it's tokens as REMI and conduct a repetition_search on them
        in order to find the most repeated token structure, and we give that to model. window_size for the repetition_search
        can be adjusted with repetition_search_div param.

        @Params:
        @single_input: single input possibly from a dataloader. Expected shape is [1, n] (I should check that later on.)
        @probability: probability of retrieval. If set to higher probability of retrieving from the user_created_dataset
        will be increased. Should be in range [0,1)
        @model: model to use for generation.
        @repetition_search_div: window_size for the repetition_search. Will use len(retrieved_tokens) // repetition_search_div

        Return:
        Generated sample with the retrieved data. If retrieval didn't happened, torch.tensor([[]]) is returned.

        Note:
        Uses self.index, so please define the index before using it. Otherwise will throw error.
        """

        if self.index == None:
            lg.critical(
                "self.index is None, cannot be used for similarity search in generation. Please specify it."
            )
            raise Exception

        if self._possibility(probability=probability):
            lg.info("Retrieving from similar songs now.")
            retrieved_tokens = self.retrieve_similar_song(
                single_input.tolist(), self.index
            )
            lg.debug(f"Retrieved tokens: {retrieved_tokens}")

            should_we, repetitive_tokens = self._repetition_search(
                tokens=retrieved_tokens,
                search_window_size=len(retrieved_tokens)
                // repetition_search_div,  # could be changed
                max_repeat=2,
            )
            if should_we:
                return model.generate(
                    inputs=torch.unsqueeze(torch.tensor(repetitive_tokens), 0).to(
                        model.device
                    ),
                    generation_config=generation_config,
                ), torch.unsqueeze(
                    torch.tensor(repetitive_tokens), 0
                )  # (N,T) LongTensor
            else:
                # if we can't find a repetitive pattern, we'll use the retrieved_tokens[:len(single_input) // repetition_search_div]
                # many tokens to generate new data.
                _input_to_model = torch.unsqueeze(
                    torch.tensor(
                        retrieved_tokens[: len(retrieved_tokens) // repetition_search_div]
                    ),
                    0,
                )
                return (
                    model.generate(
                        inputs=_input_to_model.to(model.device),
                        generation_config=generation_config,
                    ),
                    _input_to_model,
                )  # (N,T) LongTensor
        else:
            return model.generate(
                inputs=single_input.to(model.device),
                generation_config=generation_config,
            ), torch.tensor(
                [[]]
            )  # (N,T) LongTensor

    def _tweak_generation_params(
        self, p_change: float = 0.1, k_change: int = 2, temperature_change: float = 0.1
    ) -> None:
        """Tweaks the generation params so slightly to produce newer/different results."""
        self.generation_config.top_p += p_change
        self.generation_config.top_k += k_change
        self.generation_config.temperature += temperature_change
        lg.info(
            f"Tweaked the generation parameters (top_p, top_k, temperature) by {p_change},{k_change},{temperature_change}"
        )

    def _print_gen_and_retrieved(self, print_holder: List[Tuple[List[int], int]]):
        """Prints the prompt, generated and retrieved tokens in a colorfull way as the final result.
        @Params:
        @print_holder: holds the data with their corresponding class_num (0: prompt, 1: gen, 2: retrieved)
        """

        colors = ["blue", "green", "red"]  # prompt, gen, retrieved
        lg.success(
            f"Generation completed! Now you'll see the output with prompt as {colors[0]}, new generation as {colors[1]}, retrieved as {colors[2]}."
        )

        def format_color(color, data):
            return f"<{color}>{data}</>"

        final_message = ""
        for data, class_num in print_holder:
            final_message += "" + format_color(colors[class_num], data[1:-1])

        lg.opt(colors=True).success(final_message)

    def cont_gen_retrieval(
        self,
        dataloader: DataLoader,
        iteration_num: int,
        data_feed_num: int,
        model: GPT2LMHeadModel,
        probability: float,
        repetition4param_change: int = 6,
        save: bool = True,
        midi_name: Optional[str] = None,
    ):
        """Continuosly generate with retrieval. Will iterate iteration many times and will feed data_feed_num many data at each iteration.
        @Params:
        @dataloader: dataloader to use.
        @iteration: will iterate iteration many times.
        @data_feed_num: number of tokens to feed at each iteration including the first one.
        @model: model to use.
        @probability: probability
        @repetition4param_change: search for repetitive tokens. If the repetition is too much change the parameters
        to generate different data.
        @save: whether to save the midi file or not.
        @midi_name: saved midi name.
        """

        def retr_pure(tensor: torch.tensor):
            """Shape is expected to be 2D."""
            return torch.squeeze_copy(tensor).tolist()

        # lg.debug(dataloader)
        all_inputs = []

        for elem in dataloader:
            all_inputs.append(elem)

        # lg.debug(f"All inputs: {all_inputs}")
        single_input: torch.Tensor = all_inputs[0]
        # lg.debug(f"Single input: {single_input}")

        full_tokens = torch.Tensor([[-444]])  # dummy tensor

        print_holder: List[List[int]] = []
        print_holder.append([retr_pure(single_input), 0])

        # We generate first datas and then start to use iterate.
        first_result = model.generate(
            inputs=single_input.to(model.device),
            generation_config=generation_config,
        )  # (N,T) LongTensor

        lg.debug(f"First Result:\n{first_result}")
        lg.debug(f"First Results dtype:\n{first_result.dtype}")

        full_tokens = torch.cat([full_tokens, first_result], 1).to(torch.int64)  # type: ignore
        print_holder.append([retr_pure(first_result), 1])

        # lg.debug(f"New input from full tokens: \n {full_tokens[:, -data_feed_num:]}")

        for i in tqdm(range(iteration_num), desc="Generating the song! "):
            result, retrieved_result = self.generate_with_retrieval_probability(
                single_input=full_tokens[:, -data_feed_num:],
                probability=probability,
                model=model,
            )
            lg.debug(f"Here is the result of the {i}. step: {result}")

            if self._repetition_search(
                retr_pure(result), len(result) // 8, repetition4param_change
            )[0]:
                self._tweak_generation_params()

            if retrieved_result.shape != [1, 0]:  # which means did retrieved.
                full_tokens = torch.cat([full_tokens, retrieved_result, result], 1).to(
                    torch.int64
                )
                print_holder.append([retr_pure(retrieved_result), 2])
                print_holder.append([retr_pure(result), 1])

            else:  # #which means didn't retrieved.
                full_tokens = torch.cat([full_tokens, result], 1).to(torch.int64)
                print_holder.append([retr_pure(result), 1])

        self._print_gen_and_retrieved(print_holder)

        return self.midi_out(
            # not giving the dummy value with [..., 1:] statement below
            full_tokens[..., 1:].tolist(),
            self.tokenizer,
            self.dir_dict["samples"],
            "generation" if midi_name == None else midi_name,
            save,
            True,
        )

    def _repetition_search(
        self, tokens: List[int], search_window_size: int = 6, max_repeat: int = 3
    ) -> Tuple[bool, List[int]]:
        """Looks at the repetition patterns.
        @Params:
        @tokens: tokens to search in.
        @search_window_size: looks at every search_window_size length pieces.
        @max_repeat: if max repeated pattern is more than max_repeat will return True.

        Returns:
         bool: if there are more than max_repeat repeats, return True (meaning 'change the params'). Otherwise False.
         int: number of max repetition.
        """
        window_dict: Dict[Tuple[int], int] = {}

        for i in range(0, len(tokens) - search_window_size):
            # unfortunatelly lists aren't hashable. Using tuple!
            curr_data = tuple(tokens[i : i + search_window_size + 1])

            if type(curr_data[0]) == list:
                lg.critical("Well, tokens is 2D or more somehow.")
                raise Exception

            if curr_data in list(window_dict.keys()):
                window_dict[curr_data] += 1
            else:
                window_dict[curr_data] = 1

        lg.trace(f"window_dict at the end of the repetition search: \n{window_dict}")

        def _get_max_from_dict_value(a: Dict[Tuple[int], int]):
            return list(a.keys())[list(a.values()).index(max(a.values()))]

        if max(window_dict.values()) >= max_repeat:
            return (True, list(_get_max_from_dict_value(window_dict)))
        else:
            return (False, list(_get_max_from_dict_value(window_dict)))

    def retrieve_similar_song(
        self,
        search_input: List[int],
        FAISS_index: faiss.IndexFlatIP,
    ) -> List[int]:
        """Retrive the most similar data from the user defined dataset using FAISS and an
        architecture (encoder or jsymbolic-based).

        @Params:
        @search_input: input as REMI tokens.
        @index: index to use in similarity search. Right now we are supporting cos index only.
        (normalized vectors)

        Returns:
        The most similar pieces tokens in REMI.
        """

        if self.similarity_searcher == None:
            # This means that we have already created the embeddings.
            # And we just want to use search functions.
            self.similarity_searcher = SimilaritySearch(None)

        prompt_processor = self.prepare_processor(
            dataPath=None, dataItself=[self.REMI2MIDI2CP(search_input)]
        )

        input_embeddingsMidiBERT = prompt_processor[0]

        # here indices is in the dimension of [1,k].
        distances, indices = self.similarity_searcher.cos_dist(
            # First [0] is basicly the index
            # Second [0] is the embedding in the Tuple
            inputTensor=input_embeddingsMidiBERT[0][0],
            k=2,
            FAISS_index=FAISS_index,
        )

        similar_CP = self.user_dataset_MidiBERTembeds_CPtokens_ALL[indices[0][0]][
            1
        ].tolist()
        lg.trace(f"Similar CP data: {similar_CP}")
        lg.trace(f"Similar CP datas shape: {np.array(similar_CP).shape}")
        
        # returning tokens in REMI but not the embeddings.
        # we are expecting the similar_CP to be in the shape [1,512,4]
        return self.CP2MIDI2REMI(similar_CP[0])

    def _it_append(self, list_to: List[int], list_from: List[int]) -> List[int]:
        """Append integers form list_from to list_to."""
        for i in list_from:
            list_to.append(i)
        return list_to

    def generate_without_retrieval(
        self,
        dataloader: DataLoader,
        model: GPT2LMHeadModel,
    ):
        """Generate the MIDI file without applying retrieval. This function is
        mostly for debug of generation using the pretrained model.
        Generated midis will be saved in batches. And in each saved midi file will have 3 instruments which
        1) only continuation of the given prompt (current batch)
        2) only prompt
        3) both

        @Params:
        @dataloader: a dataloader to batch the prompts.
        @model: pretrianed model to use while generating.
        """
        count = 0

        for batch in tqdm(dataloader, desc="Generating Without Retrieval"): 
            result = model.generate(
                batch.to(model.device),
                generation_config=generation_config,
            )  # shape: (1,n)

            for prompt, continuation in zip(batch, result):
                generated = continuation[len(prompt) :]

                # here we are detokenizing the 3 forms of the generation.
                # list compressed as sequences of different lengths
                tokens = [generated, prompt, continuation]
                tokens = [seq.tolist() for seq in tokens]

                midi_container = tokenizer.tokens_to_midi(
                    deepcopy(tokens), time_division=384
                )
                midi_container.instruments[
                    0
                ].name = f"Continuation of original sample ({len(generated)} tokens)"
                midi_container.instruments[
                    1
                ].name = f"Original sample ({len(prompt)} tokens)"
                midi_container.instruments[2].name = f"Original sample and continuation"

                midi_container.dump(self.dir_dict["samples"] / f"container_{count}.mid")

                tokenizer.save_tokens(tokens, self.dir_dict["samples"] / f"{count}.json")
                count += 1

    def generate_continuously_without_retrieval(
        self,
        dataloader: DataLoader,
        iteration_num: int,
        data_feed_num: int,
        model: GPT2LMHeadModel,
        save: bool = True,
        midi_name: Optional[str] = None,
    ):
        """Generate the MIDI file continuously without applying retrieval. This function is
        mostly for debug of generation using the pretrained model.

        @Params:
        @dataloader: a dataloader. It's expected a single data in the dataloader. If there are multiple
        datas in the dataloader only the first one will be used.
        @iteration_num: number of iteration
        @data_feed_num: number of data to feed to model at each iteration starting after the first generation.
        @model: pretrained model to use while generating.
        @save: whether to save or not.
        @midi_name: name of the midi without .mid at the end.
        """

        # lg.debug(dataloader)
        all_inputs = []

        for elem in dataloader:
            all_inputs.append(elem)

        # lg.debug(f"All inputs: {all_inputs}")
        single_input = all_inputs[0]  # trying stuff
        # lg.debug(f"Single input: {single_input}")

        full_tokens: torch.Tensor = torch.Tensor([[-444]])  # dummy tensor

        # We generate first datas and then start to use iterate.
        first_result = model.generate(
            inputs=single_input.to(model.device),
            generation_config=generation_config,
        )  # (1,n) LongTensor
        # lg.debug(f"First Result:\n{first_result}")
        # lg.debug(f"First Results dtype:\n{first_result.dtype}")

        full_tokens = torch.cat([full_tokens, first_result], 1).to(torch.int64)
        # lg.debug(f"Full tokens (prompt+first_gen):\n{full_tokens}")

        # lg.debug(f"New input from full tokens: \n {full_tokens[:, -data_feed_num:]}")

        for i in range(iteration_num):
            result = model.generate(
                inputs=full_tokens[:, -data_feed_num:].to(model.device),
                generation_config=generation_config,
            )  # (1,n) LongTensor

            full_tokens = torch.cat([full_tokens, result], 1).to(torch.int64)

        lg.info("Generation finished!")
        # lg.debug(f"This is the final data which I'm giving to miditok:\n{full_tokens[:,1:].tolist()}")

        return self.midi_out(
            # not giving the dummy value with [:, 1:] statement
            full_tokens[..., 1:].tolist(),
            self.tokenizer,
            self.dir_dict["samples"],
            "generation" if midi_name == None else midi_name,
            save,
            True,
        )

    def midi_out(
        self,
        output_list: List[int],
        tokenizer: MIDITokenizer,
        output_path: Union[str, Path],
        midi_name: Optional[Union[str, Path]],
        save=False,
        check_err=True,
    ) -> MidiFile:
        """Saves the midi file."""

        midi_out = tokenizer.tokens_to_midi(output_list, [(24, False)])

        lg.info(f"Total of {len(output_list)} tokens will be saved.")
        if save:
            midi_out.dump(Path(output_path) / Path(f"{midi_name}.mid"))
            lg.info(f"Midi saved to {Path(output_path) / Path(f'{midi_name}.mid')}")

        if check_err:
            lg.info(
                "Token types error rate is: {}".format(
                    tokenizer.tokens_errors(output_list)  # consider_pad=True
                )
            )

        return midi_out

    def _create_directories(
        self,
        main_dir: Union[str, Path],
        dir_dict: Dict[str, Optional[Union[str, Path]]],
    ) -> Dict[str, Path]:
        """Creates the required directories using self.main_output_dir. This function is used in self.__init__()
        for createing the self.dir_dict for the first time.

        Returns a dictionary that consists of the names and the created path.
        
        @Params:
        @main_dir: main directory to craete from
        @dir_dict: names of the folders to create in the main_dir.
        """

        main_dir = Path(main_dir)

        for name in list(dir_dict.keys()):
            if Path(main_dir / Path(name)).exists() == False:
                Path(main_dir / Path(name)).mkdir(exist_ok=True)
                lg.info(f"Created: {Path(main_dir / Path(name))}")
            else:
                lg.info(f"Already exists: {Path(main_dir / Path(name))}")
            dir_dict[name] = Path(main_dir / Path(name))
        return dir_dict

    def _possibility(self, probability: float, sim_num: int = 20) -> bool:
        """Returns True with the probability of *probability*.
        @Params
        @probability: a float in range [0,1). Set this higher if you want the output to be True.
        @sim_num: how many similation to do.
        """

        x = []

        for i in range(sim_num):
            if random.random() < probability:
                x.append(1)
            else:
                x.append(0)

        if random.choice(x) == True:
            return True
        else:
            return False


In [ ]:
lg.configure(handlers=[{"sink": sys.stderr, "level": "INFO"}])

[2]

In [ ]:
dir_dict = {
            "user_defined_dataset": MAIN_PATH / Path("user_defined_dataset/"), # folder of user-defined MIDI files
            "samples": MAIN_PATH / Path("samples/"), # folder of the generated midi files.
            "index": MAIN_PATH / Path("index/Cosindex.index"), # NOT a folder, files path of the index.
            "user_dataset_CP": MAIN_PATH / Path("user_dataset_CP/"), # folder to put CP tokenized data
            "user_dataset_CP_npy_path": MAIN_PATH / Path("user_dataset_CP/CP_tokenized_user_dataset.npy"), # NOT a folder, it's the files' path.
            "embeddings": MAIN_PATH / Path("embeddings/"), # folder of the embeddings
            "user_dataset_MidiBERTembeds_CPtokens_ALL" : MAIN_PATH / Path("embeddings/user_dataset_MidiBERTembeds_CPtokens_ALL.npy"), # NOT a folder, it's the files' path.
        }

generator = GenerateWithRetrieval(
    main_output_path=MAIN_PATH,
    inference_data=dataloader_test,
    generation_config=generation_config,
    tokenizer=tokenizer,
    dir_dict=dir_dict,
    )

generated_midi_name = "" # no need to include .mid at the end.

# Below are 3 important functions. Please refer to GenerateWithRetrieval class and the functions description on how to use them.

# generator.prepare_retrieval()

#generated_midi = generator.generate_continuously_without_retrieval(
#    dataloader_test,
#    5,
#    128,
#    model,
#    generated_midi_name,
#    True,
#)

# generated_midi = generator.cont_gen_retrieval(
#     dataloader=dataloader_test,
#     iteration_num=3,
#     data_feed_num=16,
#     model=model,
#     probability=.8,
#     repetition4param_change=6,
#     save=True,
#     midi_name=generated_midi_name,
# )

generation_output_path = generator.dir_dict["samples"] / Path(generated_midi_name+".mid")


##Statistics About the Created MIDI File
Using `DisplayStats()` class, listen the generated piece and display the statistics.

In [ ]:
#@title Saving the midi by hand if needed
#generated_midi.dump(generation_output_path)

In [ ]:
displayer = DisplayStats(generation_output_path, all_data=True)
displayer.display_audio()